In [1]:
import pandas as pd
import logging
from marginal_emissions.utils.helper import search_df, check_encoding
logging.basicConfig(level=logging.INFO, format='[%(levelname)s][%(asctime)s][%(filename)s]_%(message)s')

In [2]:
kws_path = '../data/raw/other/bnetza/kraftwerksliste.csv'
cdc_path = '../data/raw/other/bnetza/ZuUndRueckbau.xlsx'  # cdc = commission/decommission
wind_path = '~/Downloads/stundenwerte_FF_00282_19710801_20241231_hist/produkt_ff_stunde_19710801_20241231_00282.txt' # avgeraged by hour
wind_path2 = '~/Downloads/stundenwerte_F_00282_19490101_20241231_hist/produkt_f_stunde_19490101_20241231_00282.txt' # hourly values
ele_em = '../data/raw/electricitymaps/DE_2024_hourly.csv'

In [3]:
kws_encoding = check_encoding(path=kws_path)

# To check exact column names
kws_cols = pd.read_csv(kws_path, sep=';', skiprows=11, header=0, nrows=5, encoding='Windows-1252')

if kws_encoding:
    kws = pd.read_csv(kws_path, sep=';', skiprows=11, header=0, encoding=kws_encoding, usecols=[
        'Anzeige-Name der Stromerzeugungseinheit',
        'Anlagenbetreiber',
        'Anschlussnetzbetreiber',
        'Bundesland der Einheit',
        'Ort der Einheit',
        'Kraftwerksstatus der Einheit',
        'Energieträger',
        'Speichertechnologie',
        'Wärmeauskopplung (KWK)\n(ja/nein)',
        'Nettonennleistung (elektrische Wirkleistung) in MW',
        'Technologie der Stromerzeugung',
        'Spannungsebene'
    ]).rename(columns={
        'Anzeige-Name der Stromerzeugungseinheit': 'Name',
        'Bundesland der Einheit': 'Bundesland',
        'Ort der Einheit': 'Ort',
        'Kraftwerksstatus der Einheit': 'Status',
        'Wärmeauskopplung (KWK)\n(ja/nein)': 'KWK',
        'Nettonennleistung (elektrische Wirkleistung) in MW': 'Net_Wirkleistung',
        'Technologie der Stromerzeugung': 'Technologie'
    })

    kw_carrier = kws[kws['Energieträger'].str.contains('Speicher', case=False, na=False)].drop(columns='Technologie')
    kw_carrier_pump = kw_carrier[kw_carrier['Speichertechnologie'] == 'Pumpspeicher']
    kw_solar = search_df(kws, 'Solar').copy() # use .copy() to avoid modifying
    kw_solar.Net_Wirkleistung = kw_solar.Net_Wirkleistung.str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float) # makes float from string
    kw_wind = kws[kws['Energieträger'].str.contains('wind', case=False, na=False)].copy()
    kw_wind.Net_Wirkleistung = kw_wind.Net_Wirkleistung.str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float) # makes float from string
    kw_kwk = kws[kws['KWK'] == 'Ja']

    print("# Kraftwerke: ", len(kws))
    print("# Speicher: ", len(kw_carrier_pump))
    print("# Must-Run: ", len(kw_kwk))

    # Einzeiler für eine ganze Spalte
    # df['column_name'] = df['column_name'].str.replace('.', '', regex=False).str.replace(',', '.').astype(float)

[INFO][54411][2026-03-13 17:34:59][helper.py][check_encoding] Checking encoding of file "../data/raw/other/bnetza/kraftwerksliste.csv"...
[INFO][54411][2026-03-13 17:34:59][helper.py][check_encoding] File is encoded as "Windows-1252"
[INFO][54411][2026-03-13 17:34:59][helper.py][search_df] Found 22 rows matching 'Solar' pattern.
# Kraftwerke:  2220
# Speicher:  114
# Must-Run:  1158


In [ ]:
test = pd.read_csv('../data/interim/generation_tennet_utc_202212232300_202501012245.csv', )